# Half-Jaw Tooth Segmentation Sample

In [ ]:
import os
import glob
import base64
import time
import requests
import json
import trimesh
import urllib
import numpy as np

def _create_colors():
    # 20 high contrast colors
    colors = [[230, 25, 75,255],[60, 180, 75,255],[255, 225, 25,255],\
            [0, 130, 200,255],[245, 130, 48,255],[145, 30, 180,255],[70, 240, 240,255],\
            [240, 50, 230,255],[210, 245, 60,255],[250, 190, 190,255],[0, 128, 128,255],\
            [230, 190, 255,255],[170, 110, 40,255],[255, 250, 200,255],[128, 0, 0,255],\
            [170, 255, 195,255],[128, 128, 0, 255]]
    #np.random.shuffle(colors)
    # gum color
    colors = [[255,255,255,255]] + colors
    return colors

def colored_mesh(mesh, label):
    COLORS = _create_colors()
    mcopy = mesh.copy()
    for i, l in enumerate(np.unique(label)):
        mcopy.visual.face_colors[np.where(label == l)[0]] = COLORS[i % 18]
    return mcopy

## Define call rules
Please modify the following code blocks based on the information you obtained from us.

In [ ]:
# Chohotech service request URL, sent with the API documentation.
base_url = "<service request URL>"

# Chohotech file service URL, sent with the API documentation.
file_server_url = "<service file server URL>"

# The authentication header must be passed in. Please keep the TOKEN confidential!!! If it is leaked, please contact us immediately to reset it. All tasks using this TOKEN will be charged to your account.
zh_token = "<your company's service Token, sent with the contact>" # All API calls must be authenticated with the token.

user_group = "APIClient" # User group, usually named APIClient.

# Your company's user_id, sent with the API documentation.
user_id = "<your company's user_id>"

# If you have received creds.json, it will be read directly below.
if os.path.exists('../../creds.json'):
    creds = json.load(open('../../creds.json', 'r'))
    base_url = creds['base_url']
    file_server_url = creds['file_server_url']
    zh_token = creds['zh_token']
    user_id = creds['user_id']
    print("loaded creds from creds.json")

In [ ]:
# The following request data uses the tooth segmentation + tooth axis task as an example. According to the contract signed between you and our company, you may not have permission to call this sample API.

# Define the basic part of the call JSON - the service name and user information
json_call = {
  "spec_group": "mesh-processing", # The invoked workflow group is sent along with the API documentation.
  "spec_name": "oral-seg", # The invoked workflow group is sent along with the API documentation. 
  "spec_version": "1.0-snapshot", # The invoked workflow group is sent along with the API documentation.
  "user_group": user_group,
  "user_id": user_id
}

### Define Callback URL

Callback will send a POST request to the specified URL after the workflow is completed. The request content is a JSON object with 4 fields:

1. workflow_id (str): the corresponding workflow_id
2. metadata (dict): the metadata you passed when starting the workflow
3. success (bool): whether the workflow succeeded (true for success)
4. reason (str or null): If success is true, this item will be null. Otherwise, it will be a string representing the reason for failure.

If you need a callback, please uncomment the following code block.

In [ ]:
# json_call['metadata'] = { # (optional) metadata can be added here, which will be attached to the callback information. Each item in the dictionary is limited to 128 characters
#     "case_id": "CH-123",
#     "case_name": "ABCDE"
# }
# json_call['notification'] =[ # (optional) callback URL can be added here
#     {"url": "https://www.baidu.com"} # multiple callback URLs can be added here, each in the format {"url": "xxx"}
# ]

## Define Input/Output Blocks

The input block is defined by `input_data` in the JSON file. For 3D meshes, we support the following **input** file formats:

```
"obj"
"stl"
"off"
"ply"
"glb"
"zip": There must be only one mesh file inside the zip, and its extension must be obj, stl, off, ply, or glb.
"tar.gz": There must be only one mesh file inside the tar or tar.gz, and its extension must be obj, stl, off, ply, or glb.
"drc"
```

and the following **output** file formats:
```
"obj"
"stl"
"off"
"ply"
"glb"
"drc"
```


We **strongly recommend** using the `drc` file format for input/output to reduce network transmission time. The generation and reading of drc format files can be done using Google's Draco library: https://github.com/google/draco

The table below shows the file size of the same 3D mesh in different file formats (point precision: 0.001mm):

| File Format | File Size |
| --- | --- |
| stl | 16.7M |
| obj | 13.5M |
| off | 14.2M |
| ply | 7.0M |
| glb | 6.0M |
| drc | 517.3K |


For 3D meshes, input can be passed in two ways:

1. Upload the 3D mesh file to our file service system first, then pass the file pointer to the request body and call the API.
2. Directly pass the base64-encoded binary data in the request body.

For 3D meshes, output can be specified in two ways:

1. Return the file pointer as part of the API response, and users can download the specific binary file from our file system.
2. Directly return the binary data in the form of base64 encoding in the API response.


We **strongly recommend** using the first method for the following reasons:

1. Base64 encoding will increase the bandwidth and increase network latency.
2. To ensure API performance, Choho may reject API requests that are too large. Therefore, for large files, the second method will fail (HTTP CODE 413).

### Upload files

In [ ]:
data = open('../../data/lower_jaw_scan.ply', 'rb').read()
resp = requests.get(file_server_url + f"/scratch/{user_group}/{user_id}/upload_url?" +
                    "postfix=ply", # Must specify postfix, i.e., file extension
                    headers={"X-ZH-TOKEN": zh_token}) # Get signed upload URL
resp.raise_for_status()

upload_url = resp.text[1:-1] # Returns a single string JSON "string", can also use json.loads(resp.text)

resp = requests.put(upload_url, data) # No auth header is needed for uploading to the cloud storage service

resp.raise_for_status()
path = "/".join(urllib.parse.urlparse(upload_url).path.lstrip("/").split("/")[3:])
urn = f"urn:zhfile:o:s:{user_group}:{user_id}:{path}"
print("file pointer:", urn)

json_call["input_data"] = { # Input data. The content varies depending on the task dictionary. Refer to the API documentation for details. This example uses tooth segmentation + tooth axis as an illustration.
    "mesh": {"type":"ply", "data": urn}, 
    "jaw_type": 'Lower' # "Lower" means lower jaw, "Upper" means upper jaw
}

# Define output configuration (output and input formats do not necessarily need to match;
# for example, input can be an STL file while output can be DRC in base64 format
json_call['output_config'] = {
    "mesh": {"type": "ply"} # If there is no 'output-location' field, a file pointer is returned by default
}

### Using Base64 for Upload

If needed, please uncomment the following code block.

In [ ]:
# json_call["input_data"] = { # Input data. The content varies depending on the task dictionary. Refer to the API documentation for details. This example uses tooth segmentation + tooth axis as an illustration.
#     "mesh": {"type":"stl", "data": base64.b64encode(open('l.stl', 'rb').read()).decode()}, # File encoded in Base64
#     "jaw_type": 'Lower' # Upper jaw is 'Upper', lower jaw is 'Lower'
# }

# # Define output configuration (output and input formats do not necessarily need to match; for example, input can be an STL file while output can be DRC in base64 format)
# json_call['output_config'] = {
#     "mesh": {"type": "ply", "output-location": "bytes"} # Download also uses Base64, obtained directly from the interface
# }

# Submit request

Submit the request using the `/run` POST method. 

In [ ]:
headers = {
  "Content-Type": "application/json",
  "X-ZH-TOKEN": zh_token
}

url = base_url + '/run'

response = requests.request("POST", url, headers=headers, data=json.dumps(json_call))
response.raise_for_status()
create_result = response.json()
run_id = create_result['run_id']

print(run_id)

# Wait for the request to complete

Use the polling API to wait for the request to complete, the API URL is `/run/{run_id}`.

**It is strongly recommended to use the callback method.** Refer to [Define Callback URL](#Define-Callback-URL), the callback method will send a callback message to the given URL at the end of the workflow, regardless of whether the workflow is successful or not, so you do not need to poll the result.

In [ ]:
# Step 2： Polling the status
url = base_url + f"/run/{run_id}"

start_time = time.time()
while time.time()-start_time < 180: # Maximum wait time: 3 minutes
    time.sleep(3) # Polling interval
    response = requests.request("GET", url, headers=headers)
    result = response.json()
    if result['completed'] or result['failed']:
        break
    
if not result['completed']:
    if result['failed']:
        raise ValueError("API error, reason： " + str(result['reason_public']))
    raise TimeoutError("API timeout")

print("API execution time： {}s".format(time.time()-start_time))

# Get the request result

Use `/data/{run_id}` to get all output data of this workflow, use `/data/{run_id}/{key}` to get a specific item of data.

In [9]:
url = base_url + f"/data/{run_id}"
response = requests.request("GET", url, headers=headers)
result = response.json()

In [ ]:
if result['mesh']['data'][:3] == "urn": # Returned via URN method, need to fetch from the file system
    resp = requests.get(file_server_url + f"/file/download?" + urllib.parse.urlencode({
                        "urn": result['mesh']['data']}),
                        headers={"X-ZH-TOKEN": zh_token})
    mesh_bytes = resp.content
else: # Returned via bytes method, read directly
    mesh_bytes = base64.b64decode(result['mesh']['data'])
result_mesh = trimesh.load(trimesh.util.wrap_as_stream(mesh_bytes), file_type=result['mesh']['type'])

# Visualization

The following section demonstrates the output results of the oral-seg sample API. 
The code above this section is common to all APIs, while the following section is specific to the processing and display of oral-seg output.

In [ ]:
print('Tooth number list：', np.unique(result['seg_labels']))
colored_mesh(result_mesh, result['seg_labels']).show()